# 04 - Doublets, ambient RNA and the clean object

Assembles the per-sample scDblFinder + decontX results (heavy, notebook 10; record linked) and builds the clean object.

Decisions (2026-11-11):
* decontX strength from the ladder test (06d): fixed prior delta (1, 300) for the 13 in vivo cell samples with non-epithelial
  Alb > 5 CP10k; raw counts for ex vivo, clean in vivo samples and all nuclei; CS140 raw + flag `ambient_dominated`.
* Scrublet calls were already removed by the draft QC, so the MASLD consensus rule (Scrublet AND scDblFinder) cannot apply;
  the 8 TotalVI r3 clusters with > 65 % scDblFinder doublets are removed (6,982 cells), as the authors removed doublet clusters.
* Re-annotation with Table S1, z-scores within cell-majority and nucleus-majority clusters separately.

Result of record: 321,059 cells and nuclei (`07_clean`).

**Runtime:** ~25 min
**Envs used (each step runs as a subprocess in its own env):** rapids-sc
**Kernel:** `masld_atlas` (orchestration only).

Outputs are written under `/mnt/masld_vault/cross_species_repro/` (mirror of `/mnt/masld_vault/cross_species/`, the outputs
of record). Inputs this notebook does not rebuild are linked from the record; see `repro_setup.py` for the safety rules.

In [ ]:
%run -i /home/ubuntu/liver_cross_species/notebooks/repro_setup.py
print("RECORD", RECORD); print("REPRO ", REPRO); print("WORK  ", WORK)

In [ ]:
guard(30)
link("guilliams2022/02_qc", "guilliams2022/03_integration", "guilliams2022/05_annotation_draft")
fresh("guilliams2022/06_doublets_ambient")
link("guilliams2022/06_doublets_ambient/per_sample", "guilliams2022/06_doublets_ambient/ladder", "guilliams2022/06_doublets_ambient/final_dx")

### decontX ladder (record of notebook 10): Alb removed from non-epithelial cells vs Alb kept in hepatocytes

In [ ]:
import pandas as pd, glob
pd.concat([pd.read_csv(f, sep="\t").assign(sample=f.split("/")[-1].split(".")[0]) for f in sorted(glob.glob(rep("guilliams2022/06_doublets_ambient/ladder/*.ladder.tsv")))])

### 06c - assemble per-sample results

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/06c_assemble.py
"""Step 06c - assemble scDblFinder + decontX results for the draft-QC cells (328,041).

Doublets (MASLD rule): consensus_doublet = Scrublet AND scDblFinder; removal only on consensus. Per-cluster (TotalVI Leiden r3)
doublet rates are reported for the authors' style cluster-level removal (decided by hand, not here).
Ambient: decontX counts per sample; Alb diagnostic = Alb CP10k in non-epithelial draft types before/after, Alb kept in
hepatocytes.
Outputs <OUT>/06_doublets_ambient/:
  guilliams2022_fig1b_decontx.h5ad  X = decontX counts, layers['counts'] = raw counts, obs flags, obsm adt / adt_measured
  cells_doublets_ambient.parquet, per_sample_summary.tsv, per_cluster_r3_summary.tsv, figures/*.png|pdf
"""
import glob
import os
import sys

import anndata as ad
import matplotlib
import numpy as np
import pandas as pd
import scipy.sparse as sp

matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = sys.argv[1] if len(sys.argv) > 1 else "/mnt/masld_vault/cross_species_repro/guilliams2022"
D = f"{OUT}/06_doublets_ambient"
os.makedirs(f"{D}/figures", exist_ok=True)

a = ad.read_h5ad(f"{OUT}/02_qc/guilliams2022_fig1b_qc.h5ad")
f = pd.read_parquet(f"{OUT}/02_qc/guilliams2022_fig1b_qc_flags.parquet")
E = pd.read_parquet(f"{OUT}/05_annotation_draft/cells_annotation_draft.parquet")
assert (E.index == a.obs_names).all()
res = pd.concat([pd.read_parquet(p) for p in glob.glob(f"{D}/per_sample/*.cells.parquet")])
assert len(res) == 389056 and res.index.is_unique

# decontX counts, kept cells in each sample's order (= raw object order = 02_qc order within sample)
blocks, names = [], []
for s, idx in a.obs.groupby("sample", observed=True).indices.items():
    m = sp.load_npz(f"{D}/per_sample/{s}.decontx.npz")
    assert m.shape[0] == len(idx), s
    blocks.append(m)
    names.append(a.obs_names[idx])
order = np.concatenate([a.obs_names.get_indexer(n) for n in names])
M = sp.vstack(blocks).tocsr()
inv = np.empty_like(order)
inv[order] = np.arange(len(order))
M = M[inv]
assert M.shape == a.shape

o = a.obs.copy()
o["scrublet_doublet"] = f.loc[o.index, "scrublet_doublet"].values
o["scDblFinder_score"] = res.loc[o.index, "scDblFinder_score"].values
o["scDblFinder_class"] = res.loc[o.index, "scDblFinder_class"].values
o["consensus_doublet"] = o.scrublet_doublet & (o.scDblFinder_class == "doublet")
o["decontX_contamination"] = res.loc[o.index, "decontX_contamination"].values
o["umis_raw"] = np.asarray(a.X.sum(1)).ravel()
o["umis_decontx"] = np.asarray(M.sum(1)).ravel()
for c in E.columns:
    if c not in o.columns:
        o[c] = E[c].values
o["annotation_draft"] = o["annotation_draft"].astype(str)
o["dx_lt200"] = o.umis_decontx < 200

# per-sample summary
g = o.groupby("sample", observed=True)
ps = pd.DataFrame({"suspension": g.suspension.first(), "digestion": g.digestion.first(), "n_kept": g.size(),
                   "scrublet": g.scrublet_doublet.mean(), "scDblFinder": g.apply(lambda x: (x.scDblFinder_class == "doublet").mean()),
                   "consensus": g.consensus_doublet.mean(), "median_contamination": g.decontX_contamination.median(),
                   "umis_kept": g.umis_decontx.sum() / g.umis_raw.sum(), "dx_lt200": g.dx_lt200.sum()})

# Alb diagnostic
sym = a.var["symbol"].astype(str).values
alb = np.flatnonzero(sym == "Alb")[0]
cp = lambda X, u: np.asarray(X[:, alb].todense()).ravel() / np.maximum(u, 1) * 1e4  # noqa: E731
o["Alb_cp10k_raw"] = cp(a.X, o.umis_raw.values)
o["Alb_cp10k_decontx"] = cp(M, o.umis_decontx.values)
epi = o.annotation_draft.isin(["Hepatocyte", "Cholangiocyte"])
hep = o.annotation_draft == "Hepatocyte"
ps["nonepi_Alb_cp10k_raw"] = o[~epi].groupby("sample", observed=True).Alb_cp10k_raw.mean()
ps["nonepi_Alb_cp10k_dx"] = o[~epi].groupby("sample", observed=True).Alb_cp10k_decontx.mean()
alb_raw = np.asarray(a.X[:, alb].todense()).ravel()
alb_dx = np.asarray(M[:, alb].todense()).ravel()
ps["hep_Alb_kept"] = pd.Series(alb_dx[hep.values], o.index[hep.values]).groupby(o.loc[hep, "sample"].values).sum() / \
    pd.Series(alb_raw[hep.values], o.index[hep.values]).groupby(o.loc[hep, "sample"].values).sum()
ps["nonepi_Alb_removed"] = 1 - pd.Series(alb_dx[~epi.values]).groupby(o.loc[~epi, "sample"].values).sum() / \
    pd.Series(alb_raw[~epi.values]).groupby(o.loc[~epi, "sample"].values).sum()
ps.sort_values(["suspension", "digestion"]).to_csv(f"{D}/per_sample_summary.tsv", sep="\t", float_format="%.4g")
print(ps.sort_values(["suspension", "digestion"]).to_string(float_format=lambda v: f"{v:.3f}"), flush=True)

# per-cluster (r3) summary for cluster-level doublet / contamination review
g = o.groupby("leiden_totalvi_r3", observed=True)
pc = pd.DataFrame({"annotation_draft": g.annotation_draft.agg(lambda x: x.mode().iat[0]), "n": g.size(),
                   "frac_nucleus": g.apply(lambda x: (x.suspension == "nucleus").mean()),
                   "scDblFinder": g.apply(lambda x: (x.scDblFinder_class == "doublet").mean()),
                   "scDblFinder_score_median": g.scDblFinder_score.median(), "consensus": g.consensus_doublet.mean(),
                   "median_contamination": g.decontX_contamination.median(), "dx_lt200": g.dx_lt200.mean()})
pc = pc.sort_values("scDblFinder", ascending=False)
pc.to_csv(f"{D}/per_cluster_r3_summary.tsv", sep="\t", float_format="%.4g")
print(pc.head(25).to_string(float_format=lambda v: f"{v:.3f}"), flush=True)

tot = {"cells": len(o), "scrublet": int(o.scrublet_doublet.sum()), "scDblFinder": int((o.scDblFinder_class == "doublet").sum()),
       "consensus": int(o.consensus_doublet.sum()), "dx_lt200": int(o.dx_lt200.sum()),
       "umis_kept": float(o.umis_decontx.sum() / o.umis_raw.sum()),
       "nonepi_Alb_cp10k_raw": float(o.Alb_cp10k_raw[~epi].mean()), "nonepi_Alb_cp10k_dx": float(o.Alb_cp10k_decontx[~epi].mean())}
print(tot, flush=True)
o.to_parquet(f"{D}/cells_doublets_ambient.parquet")

# figures: contamination and doublet score by draft type, split by suspension
types = o.groupby("annotation_draft").size().sort_values(ascending=False).index
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
for ax, col, ttl in [(axs[0], "decontX_contamination", "decontX contamination"),
                     (axs[1], "scDblFinder_score", "scDblFinder score")]:
    for j, (susp, colr) in enumerate([("cell", "tab:blue"), ("nucleus", "tab:orange")]):
        data = [o.loc[(o.annotation_draft == t) & (o.suspension == susp), col].dropna().values for t in types]
        pos = np.arange(len(types)) + (j - 0.5) * 0.4
        keep = [i for i, d in enumerate(data) if len(d) > 10]
        vp = ax.violinplot([data[i] for i in keep], positions=pos[keep], widths=0.38, showmedians=True)
        for b in vp["bodies"]:
            b.set_color(colr)
    ax.set_xticks(range(len(types)), types, rotation=90)
    ax.set_title(f"{ttl} (blue cells, orange nuclei)")
x = np.arange(len(types))
axs[2].bar(x - 0.2, [o.Alb_cp10k_raw[o.annotation_draft == t].mean() for t in types], 0.4, label="raw")
axs[2].bar(x + 0.2, [o.Alb_cp10k_decontx[o.annotation_draft == t].mean() for t in types], 0.4, label="decontX")
axs[2].set_yscale("log")
axs[2].set_xticks(x, types, rotation=90)
axs[2].set_title("Alb CP10k by draft type")
axs[2].legend()
fig.tight_layout()
fig.savefig(f"{D}/figures/doublets_ambient_by_type.png", dpi=200)
fig.savefig(f"{D}/figures/doublets_ambient_by_type.pdf")

Em = ad.read_h5ad(f"{OUT}/03_integration/embeddings.h5ad")
U = Em.obsm["X_umap_totalvi"]
fig, axs = plt.subplots(1, 4, figsize=(24, 6))
for ax, (v, ttl, kw) in zip(axs, [(o.scDblFinder_score.values, "scDblFinder score", dict(cmap="viridis")),
                                  (o.consensus_doublet.values.astype(float), "consensus doublet", dict(cmap="Reds")),
                                  (o.decontX_contamination.values, "decontX contamination", dict(cmap="magma", vmax=0.6)),
                                  (np.log1p(o.Alb_cp10k_decontx.values), "Alb log1p CP10k after decontX", dict(cmap="viridis"))]):
    ordr = np.argsort(v)
    sc_ = ax.scatter(U[ordr, 0], U[ordr, 1], c=v[ordr], s=0.2, rasterized=True, **kw)
    ax.set_title(ttl)
    ax.axis("off")
    fig.colorbar(sc_, ax=ax, shrink=0.6)
fig.savefig(f"{D}/figures/umap_doublets_ambient.png", dpi=200, bbox_inches="tight")
fig.savefig(f"{D}/figures/umap_doublets_ambient.pdf", dpi=300, bbox_inches="tight")

h = ad.AnnData(X=M, obs=o, var=a.var.copy(), layers={"counts": a.X}, obsm={k: v for k, v in a.obsm.items()})
h.obsm["X_totalvi_draft"] = Em.obsm["X_totalvi"]
h.obsm["X_umap_totalvi_draft"] = U
h.uns["note"] = ("DRAFT 2026-11-11. X = decontX counts (celda, per sample, z = TotalVI Leiden r1, defaults, varGenes 5000, seed 12345); "
                 "layers counts = raw UMI. Cells = draft QC (authors 3-MAD rule). consensus_doublet = Scrublet & scDblFinder (not removed). "
                 "var_names = Ensembl IDs (Ens93), var symbol. obsm adt = 173 ADT raw counts (see adt_measured).")
h.write_h5ad(f"{D}/guilliams2022_fig1b_decontx.h5ad.tmp", compression="gzip")
os.replace(f"{D}/guilliams2022_fig1b_decontx.h5ad.tmp", f"{D}/guilliams2022_fig1b_decontx.h5ad")
print("done", h.shape, flush=True)

In [ ]:
run("06c_assemble.py", "rapids-sc")

In [ ]:
compare_tsv("guilliams2022/06_doublets_ambient/per_sample_summary.tsv")

In [ ]:
fresh("guilliams2022/07_clean")

### 06f - clean object, re-annotation, cells / nuclei split

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/06f_clean_annotate_split.py
"""Step 06f - clean object, re-annotation, split into single cells and single nuclei (user go 2026-11-11).

Ambient (from the 06d ladder):
  - decontX (1, 300) for the 13 in vivo cell samples with non-epithelial Alb > 5 CP10k (06e outputs);
  - raw counts for ex vivo, the clean in vivo samples and all nuclei (decontX removed hepatocyte signal in nuclei);
  - CS140: raw counts + flag ambient_dominated (ambient Alb 440 CP10k; decontX cost hepatocytes ~30 % Alb at every strength;
    decision open, default = MASLD raw_by_decision precedent).
  - dx_lt200: < 200 UMI after decontX (decontX samples only, neutrophils exempt) - flagged, not removed.
Doublets: the draft QC already removed Scrublet calls, so the MASLD consensus rule cannot apply. Removed = the 8 TotalVI
Leiden r3 clusters with > 65 % scDblFinder doublets (authors' approach: doublet clusters removed by hand). Per-cell scDblFinder
calls elsewhere are flagged only.
Annotation: Table S1 (mmc1) top-50 mouse DEGs, cluster-level (Leiden r3), log-normalised cleaned counts; z-scores computed
separately within cell-majority and nucleus-majority clusters (mixing them mislabelled hepatocyte nuclei in the draft).
Outputs <OUT>/07_clean/: guilliams2022_fig1b_clean.h5ad (all), _cells.h5ad, _nuclei.h5ad, cluster_labels_clean.tsv,
composition_clean.tsv, label_changes.tsv, figures/.
"""
import os

import anndata as ad
import matplotlib
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = "/mnt/masld_vault/cross_species_repro/guilliams2022"
DX = f"{OUT}/06_doublets_ambient/final_dx"
D = f"{OUT}/07_clean"
os.makedirs(f"{D}/figures", exist_ok=True)
S1 = "/mnt/masld_vault/geo/GSE192740/paper_supplement_cell2022/1-s2.0-S0092867421014811-mmc1.xlsx"
TIER1 = "CS114 CS137 CS138 CS139 CS144 CS48 CS52 CS88 CS89 CS92 CS93 CS96 CS97".split()
DOUBLET_CLUSTERS = ["45", "41", "50", "12", "5", "71", "23", "39"]
NTOP = 50

a = ad.read_h5ad(f"{OUT}/06_doublets_ambient/guilliams2022_fig1b_decontx.h5ad")
raw = a.layers["counts"].tocsr()
obs = a.obs.copy()
obs["decontx_setting"] = "raw"
obs["decontX_contamination_final"] = np.nan
rows, blocks = [], []
for s in TIER1:
    idx = np.flatnonzero(obs["sample"].astype(str).values == s)
    m = sp.load_npz(f"{DX}/{s}.dx300.npz").tocsr()
    c = pd.read_parquet(f"{DX}/{s}.dx300.parquet")
    assert m.shape[0] == len(idx) and (c.index == obs.index[idx]).all(), s
    rows.append(idx)
    blocks.append(m)
    obs.iloc[idx, obs.columns.get_loc("decontx_setting")] = "decontX_1_300"
    obs.iloc[idx, obs.columns.get_loc("decontX_contamination_final")] = c.iloc[:, 0].values
# assemble final counts: raw rows except tier-1 rows replaced
repl = np.concatenate(rows)
keep_raw = np.setdiff1d(np.arange(a.n_obs), repl)
parts = sp.vstack([raw[keep_raw].astype(np.float32), sp.vstack(blocks).astype(np.float32)]).tocsr()
order = np.concatenate([keep_raw, repl])
inv = np.empty_like(order)
inv[order] = np.arange(len(order))
X = parts[inv]
obs["umis_final"] = np.asarray(X.sum(1)).ravel()
obs["ambient_dominated"] = obs["sample"].astype(str) == "CS140"
obs["dx_lt200"] = (obs.decontx_setting != "raw") & (obs.umis_final < 200) & (obs.annotation_draft != "Neutrophil")
obs["scDblFinder_doublet"] = obs.scDblFinder_class == "doublet"
obs["doublet_cluster"] = obs.leiden_totalvi_r3.astype(str).isin(DOUBLET_CLUSTERS)
print("cells", len(obs), "doublet-cluster removed", int(obs.doublet_cluster.sum()), "dx_lt200 flagged", int(obs.dx_lt200.sum()),
      "CS140 flagged", int(obs.ambient_dominated.sum()), flush=True)

c = ad.AnnData(X=X, obs=obs, var=a.var.copy(), layers={"counts": raw}, obsm=dict(a.obsm))
c = c[~c.obs.doublet_cluster.values].copy()
del a, raw, X, parts

# re-annotation on cleaned counts
n = c.copy()
sc.pp.normalize_total(n, target_sum=1e4)
sc.pp.log1p(n)
sym = n.var["symbol"].astype(str).values
x = pd.read_excel(S1, sheet_name=None)
markers = {}
for k, v in x.items():
    if k.startswith("Mouse") and k.endswith("DEGs"):
        g = v.sort_values("score", ascending=False).iloc[:NTOP, 0].astype(str)
        markers[k.replace("Mouse ", "").replace(" DEGs", "")] = [s for s in g if s in set(sym)]
cl = n.obs.leiden_totalvi_r3.astype(str).values
cats = sorted(set(cl), key=int)
codes = pd.Categorical(cl, categories=cats).codes
ind = sp.csr_matrix((np.ones(len(cl)), (codes, np.arange(len(cl)))), shape=(len(cats), len(cl)))
ncl = np.asarray(ind.sum(1)).ravel()
need = sorted({s for v in markers.values() for s in v})
cols = np.where(np.isin(sym, need))[0]
G = pd.DataFrame(np.asarray((ind @ n.X[:, cols]).todense()) / ncl[:, None], index=cats, columns=sym[cols]).T.groupby(level=0).mean().T
fnuc = pd.Series((n.obs.suspension.astype(str).values == "nucleus"), index=n.obs_names).groupby(cl).mean().reindex(cats)
grp = np.where(fnuc.values > 0.5, "nucleus", "cell")
score = pd.DataFrame(index=cats, columns=list(markers), dtype=float)
for gname in ("cell", "nucleus"):
    sel = grp == gname
    Gs = G.loc[sel]
    Z = (Gs - Gs.mean()) / Gs.std().replace(0, np.nan)
    for t_, gl in markers.items():
        score.loc[Gs.index, t_] = Z[gl].mean(1)
score.to_csv(f"{D}/cluster_scores_clean.tsv", sep="\t", float_format="%.4g")
srt = np.sort(score.values, 1)
lab = pd.DataFrame({"n_cells": ncl.astype(int), "z_group": grp, "frac_nucleus": fnuc.values, "label": score.idxmax(1),
                    "score": srt[:, -1], "second": score.apply(lambda r: r.drop(r.idxmax()).idxmax(), axis=1),
                    "margin": srt[:, -1] - srt[:, -2]}, index=cats)
lab["low_margin"] = lab.margin < 0.5
draft = c.obs.groupby(c.obs.leiden_totalvi_r3.astype(str), observed=True).annotation_draft.agg(lambda s: s.mode().iat[0])
lab["label_draft"] = draft.reindex(cats).values
lab.to_csv(f"{D}/cluster_labels_clean.tsv", sep="\t", float_format="%.4g")
lab[lab.label != lab.label_draft].to_csv(f"{D}/label_changes.tsv", sep="\t", float_format="%.4g")
print(lab.to_string(float_format=lambda v: f"{v:.2f}"), flush=True)
print("label changes draft -> clean:\n", lab[lab.label != lab.label_draft][["n_cells", "frac_nucleus", "label_draft", "label",
                                                                              "margin"]].to_string(), flush=True)
del n

c.obs["annotation_clean"] = pd.Categorical(lab.loc[cl, "label"].values)
c.obs["annotation_clean_low_margin"] = lab.loc[cl, "low_margin"].values
comp = c.obs.groupby(["annotation_clean", "suspension"], observed=True).size().unstack(fill_value=0)
comp["total"] = comp.sum(1)
comp = comp.sort_values("total", ascending=False)
comp.to_csv(f"{D}/composition_clean.tsv", sep="\t")
print(comp.to_string(), "\ntotal", int(comp.total.sum()), flush=True)
for k in list(c.obs.columns):
    if c.obs[k].dtype == object:
        c.obs[k] = c.obs[k].astype(str)
c.uns["note"] = ("DRAFT-CLEAN 2026-11-11. X = final counts (decontX (1,300) for 13 in vivo cell samples, raw otherwise; "
                 "obs decontx_setting); layers counts = raw UMI. Removed: 8 doublet clusters (TotalVI Leiden r3, >65 % scDblFinder). "
                 "Flags kept: scDblFinder_doublet, dx_lt200, ambient_dominated (CS140, raw). annotation_clean = Table S1 top-50 DEGs, "
                 "cluster level, z within cell / nucleus clusters. Embedding X_totalvi_draft = TotalVI on uncleaned counts.")
c.write_h5ad(f"{D}/guilliams2022_fig1b_clean.h5ad", compression="gzip")
print("written clean", c.shape, flush=True)

import rapids_singlecell as rsc  # noqa: E402

for susp, tag in (("cell", "cells"), ("nucleus", "nuclei")):
    s = c[c.obs.suspension.astype(str).values == susp].copy()
    t = ad.AnnData(obs=s.obs[[]].copy(), obsm={"X": s.obsm["X_totalvi_draft"].astype(np.float32)})
    rsc.pp.neighbors(t, use_rep="X")
    rsc.tl.umap(t, random_state=0)
    s.obsm["X_umap"] = np.asarray(t.obsm["X_umap"])
    if susp == "nucleus":
        for k in ("adt", "adt_measured"):
            s.obsm.pop(k, None)
    s.write_h5ad(f"{D}/guilliams2022_fig1b_clean_{tag}.h5ad", compression="gzip")
    fig, axs = plt.subplots(1, 2, figsize=(16, 7))
    sc.pl.umap(s, color="annotation_clean", ax=axs[0], show=False, legend_loc="on data", legend_fontsize=6, frameon=False,
               title=f"{tag}: {s.n_obs:,} (draft TotalVI latent, cleaned labels)", s=2)
    sc.pl.umap(s, color="digestion", ax=axs[1], show=False, frameon=False, s=2)
    fig.savefig(f"{D}/figures/umap_clean_{tag}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("written", tag, s.shape, flush=True)
print("done", flush=True)

In [ ]:
run("06f_clean_annotate_split.py", "rapids-sc", gpu=0)

In [ ]:
compare_tsv("guilliams2022/07_clean/cluster_labels_clean.tsv")
compare_tsv("guilliams2022/07_clean/composition_clean.tsv")